# NB01 — Pattern Generation
結構光嘅靈魂問題：投影儀邊一列照亮咗場景嘅呢一點？
答案係**用光本身做編碼**：一組精心設計嘅圖案，每個 pixel 睇完就知自己俾邊列照住。
呢課生成 SLMaster 用嘅兩種圖案：**相位平移正弦波**（亞像素精度）+ **格雷碼**（消歧義）。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'


## 1. 相位平移正弦波（phase-shifted sinusoids）
投影儀播 N 張正弦條紋，每張相位錯開 2π/N。相機睇同一點喺 N 張入面嘅亮度變化，
就可以用 atan2 還原出佢喺正弦週期入面嘅**相位**（精度 ~1/100 週期）。

In [2]:
from sl_edu import patterns

imgs = patterns.generate(width=1920, height=1080, shift_time=4, n_periods=32)
print(f"{len(imgs)} patterns: 4 phase + 5 gray")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for k, ax in zip(range(3), axes):
    ax.imshow(imgs[k], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'phase shift {k}/4'); ax.axis('off')
plt.show()

# 一個週期入面嘅亮度曲線（row 540, 頭 120 列）
for k in range(4):
    plt.plot(imgs[k][540, :120], label=f'I{k}')
plt.legend(); plt.xlabel('projector column'); plt.ylabel('intensity')
plt.title('4-step phase shift'); plt.show()

9 patterns: 4 phase + 5 gray


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82679/1416492697.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82679/1416492697.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('4-step phase shift'); plt.show()


## 2. 格雷碼（Gray code）
相位只話到「週期內第幾」，唔知「第幾個週期」。格雷碼用 ⌈log₂(32)⌉=5 張黑白圖
俾每個週期一個 5-bit 門牌號碼。關鍵性質：**相鄰週期只差 1 bit**，邊界誤判最多錯一格。
SLMaster 仲會將格雷碼平移半個週期（「shift」），令佢嘅邊界避開正弦波嘅 wrap 邊界——
兩套編碼嘅危險位永遠唔重疊。

In [3]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for k, ax in enumerate(axes):
    ax.imshow(imgs[4 + k], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'gray bit {k} (MSB first)'); ax.axis('off')
plt.show()

# 解讀 row 540：邊度係週期邊界？
bits = np.stack([(imgs[4 + k][540] > 128) for k in range(5)], axis=-1)
gray_val = np.zeros(1920, int)
for k in range(5):
    gray_val = (gray_val << 1) | bits[..., k]
plt.plot(gray_val[:400]); plt.xlabel('column'); plt.ylabel('raw gray value')
plt.title('MSB-first accumulation (before gray->binary XOR)'); plt.show()

/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82679/4278463630.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82679/4278463630.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('MSB-first accumulation (before gray->binary XOR)'); plt.show()


## 3. 點解兩樣都要？
- **淨相位**：精度高但 2π 歧義（32 個週期全部一樣樣）
- **淨格雷碼**：冇歧義但精度得 1 個週期（60 px ≈ 幾 mm 深度誤差）
- **合體**：格雷碼話你知邊個週期，相位話你知週期內位置 → 高精度 + 冇歧義

下一課：相機實際影到嘅係點？（capture & simulation）